# Experimento 4: HDBSCAN

Este notebook implementa el algoritmo HDBSCAN sobre la matriz normalizada de características de trayectorias académicas. Se evalúa su capacidad para identificar agrupaciones y observaciones atípicas mediante métricas internas de validación.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import hdbscan

from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Entorno listo")

Entorno listo


In [2]:
ruta = Path("../data/processed/X_clustering.csv")

X = pd.read_csv(ruta)

X.head()

,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
0,-0.487231,0.516412,-0.402032,-0.464573,-0.711080,-0.596566,0.147024,0.486179,0.433139,0.673603,-0.961629
1,-0.487231,-0.420977,-0.487768,0.028994,0.942910,1.321249,0.383717,-1.921609,-2.051139,-0.304771,1.363608
2,-0.487231,-0.366903,-0.487768,0.028994,0.777511,1.129467,0.265371,-1.921609,-2.009536,-0.222381,1.221827
3,-1.416016,-2.899670,-1.387992,-1.385899,0.446713,0.745904,-1.825423,-1.754733,-1.108143,-2.636562,1.369857
4,0.441553,0.189696,0.555349,0.950319,-0.545681,-0.596566,-0.629817,0.831851,1.050932,0.599907,-0.535305


In [3]:
resultados_hdbscan = []

for min_cluster_size in [5, 10, 15, 20, 30]:

    for min_samples in [3, 5, 10]:

        modelo = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples
        )

        etiquetas = modelo.fit_predict(X)


        # quitar ruido (-1)
        etiquetas_validas = etiquetas != -1


        cantidad_clusters = len(
            set(etiquetas)
        ) - (1 if -1 in etiquetas else 0)


        if cantidad_clusters >= 2:

            silhouette = silhouette_score(
                X[etiquetas_validas],
                etiquetas[etiquetas_validas]
            )

            davies = davies_bouldin_score(
                X[etiquetas_validas],
                etiquetas[etiquetas_validas]
            )

        else:
            silhouette = np.nan
            davies = np.nan


        resultados_hdbscan.append({
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "clusters": cantidad_clusters,
            "ruido": sum(etiquetas == -1),
            "silhouette": silhouette,
            "davies_bouldin": davies
        })


resultados_hdbscan = pd.DataFrame(
    resultados_hdbscan
)

resultados_hdbscan

,min_cluster_size,min_samples,clusters,ruido,silhouette,davies_bouldin
0,5,3,7,49,0.431476,0.805032
1,5,5,6,69,0.466235,0.771182
2,5,10,3,69,0.443859,0.705060
3,10,3,6,54,0.442985,0.826465
4,10,5,6,69,0.466235,0.771182
5,10,10,3,69,0.443859,0.705060
6,15,3,4,50,0.465957,0.802441
7,15,5,5,65,0.463041,0.761954
8,15,10,2,43,0.593727,0.598189
9,20,3,4,50,0.465957,0.802441


In [4]:
mejor_hdbscan = (
    resultados_hdbscan
    .sort_values(
        "silhouette",
        ascending=False
    )
    .iloc[0]
)

mejor_hdbscan

min_cluster_size    30.000000
min_samples         10.000000
clusters             2.000000
ruido               43.000000
silhouette           0.593727
davies_bouldin       0.598189
Name: 14, dtype: float64

In [5]:
modelo_final = hdbscan.HDBSCAN(
    min_cluster_size=int(mejor_hdbscan["min_cluster_size"]),
    min_samples=int(mejor_hdbscan["min_samples"])
)

clusters = modelo_final.fit_predict(X)

In [6]:
pd.Series(clusters).value_counts()

 1    166
-1     43
 0     35
Name: count, dtype: int64

In [7]:
mask = clusters != -1

silhouette_final = silhouette_score(
    X[mask],
    clusters[mask]
)

davies_final = davies_bouldin_score(
    X[mask],
    clusters[mask]
)

print("Silhouette:", silhouette_final)
print("Davies-Bouldin:", davies_final)

Silhouette: 0.5937269217213702
Davies-Bouldin: 0.5981888636116464


In [12]:
df_original = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

resultado_hdbscan = df_original.copy()

resultado_hdbscan["cluster"] = clusters

resultado_hdbscan.groupby(
    "cluster"
).mean(numeric_only=True)

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
cluster,,,,,,,,,,,,
-1,5.944864e+07,5.534884,10.425095,124.744186,35.930233,12.581395,9.139535,11.972868,7.117199,-4.855669,0.681554,4.960875
0,6.171678e+07,3.600000,9.714809,81.428571,22.000000,9.600000,8.485714,14.140476,0.902700,-13.237776,0.647340,6.405990
1,6.264286e+07,7.469880,14.639876,163.849398,55.313253,1.036145,0.415663,14.528929,14.891966,0.363037,0.982362,1.722227


In [10]:
resultado_hdbscan = pd.DataFrame({
    "modelo": ["HDBSCAN"],
    "configuracion": ["min_cluster_size=30, min_samples=10"],
    "silhouette": [silhouette_final],
    "davies_bouldin": [davies_final],
    "ruido": [43]
})


resultado_hdbscan.to_csv(
    "../results/hdbscan/evaluacion_hdbscan.csv",
    index=False,
    encoding="utf-8-sig"
)

In [13]:
caracterizacion_hdbscan = (
    resultado_hdbscan
    .groupby("cluster")
    .mean(numeric_only=True)
)


caracterizacion_hdbscan.to_csv(
    "../results/hdbscan/caracterizacion_clusters.csv",
    encoding="utf-8-sig"
)